In [ ]:
import pandas as pd
import numpy as np

In [56]:
def convert_all_to_xlsx():
    files = ["S_push_1_f", "S_push_2_nf", "S_push_3_f", "S_push_4_nf",
              "S_row_1_f", "S_row_2_nf", "S_row_3_f", "S_row_4_nf",
              "S_sq_1_f", "S_sq_2_nf", "S_sq_3_f", "S_sq_4_nf",
              "T_push_1_f", "T_push_2_nf", "T_push_3_f", "T_push_4_nf",
              "T_row_1_f", "T_row_2_nf", "T_row_3_f", "T_row_4_nf",
              "T_sq_1_f", "T_sq_2_nf", "T_sq_3_f", "T_sq_4_nf",]

    for xls_file in files:
        sheets = pd.read_excel(f'raw/{xls_file}.xls', sheet_name=None)

        with pd.ExcelWriter(f"preproc/{xls_file}.xlsx", engine="openpyxl") as writer:
            for sheet_name, df in sheets.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)

        print(f"Converted: {xls_file}")
convert_all_to_xlsx()

Converted: S_push_1_f
Converted: S_push_2_nf
Converted: S_push_3_f
Converted: S_push_4_nf
Converted: S_row_1_f
Converted: S_row_2_nf
Converted: S_row_3_f
Converted: S_row_4_nf
Converted: S_sq_1_f
Converted: S_sq_2_nf
Converted: S_sq_3_f
Converted: S_sq_4_nf
Converted: T_push_1_f
Converted: T_push_2_nf
Converted: T_push_3_f
Converted: T_push_4_nf
Converted: T_row_1_f
Converted: T_row_2_nf
Converted: T_row_3_f
Converted: T_row_4_nf
Converted: T_sq_1_f
Converted: T_sq_2_nf
Converted: T_sq_3_f
Converted: T_sq_4_nf


In [57]:
def resample_sensor(df, value_cols, time_col="Time (s)", duration=30.0, hz=50):
    df = df.sort_values(time_col).copy()

    df[time_col] = df[time_col] - df[time_col].iloc[0]

    target_t = np.arange(0, duration, 1 / hz)

    out = pd.DataFrame({time_col: target_t})

    for col in value_cols:
        out[col] = np.interp(
            target_t,
            df[time_col].to_numpy(),
            df[col].to_numpy()
        )

    return out

In [58]:
def resample_by_sensor(df_sensor, sensor):
    df_sensor_resampled = None
    if sensor == "Accelerometer":
        df_sensor_resampled = resample_sensor(
            df_sensor,
            value_cols=[
                "Acceleration x (m/s^2)",
                "Acceleration y (m/s^2)",
                "Acceleration z (m/s^2)",
            ]
        )
    elif sensor == "Gyroscope":
        df_sensor_resampled = resample_sensor(
            df_sensor,
            value_cols=[
                "Gyroscope x (rad/s)",
                "Gyroscope y (rad/s)",
                "Gyroscope z (rad/s)",
            ]
        )
    elif sensor == "Linear Acceleration":
        df_sensor_resampled = resample_sensor(
            df_sensor,
            value_cols=[
                "Linear Acceleration x (m/s^2)",
                "Linear Acceleration y (m/s^2)",
                "Linear Acceleration z (m/s^2)",
            ]
        )
    elif sensor == "Orientation":
        df_sensor_resampled = resample_sensor(
            df_sensor,
            value_cols=[
                "Yaw (°)",
                "Pitch (°)",
                "Roll (°)",
            ]
        )
    return df_sensor_resampled

In [59]:
CUTS = {
    "S_push_1_f":3,
    "S_push_2_nf":3,
    "S_push_3_f":4,
    "S_push_4_nf":3,

    "S_row_1_f":3,
    "S_row_2_nf":4,
    "S_row_3_f":3,
    "S_row_4_nf":4,

    "S_sq_1_f":3,
    "S_sq_2_nf":4,
    "S_sq_3_f":4,
    "S_sq_4_nf":4,

    "T_push_1_f":4,
    "T_push_2_nf":4,
    "T_push_3_f":3,
    "T_push_4_nf":5,

    "T_row_1_f":5,
    "T_row_2_nf":5,
    "T_row_3_f":3,
    "T_row_4_nf":3,

    "T_sq_1_f":3,
    "T_sq_2_nf":4,
    "T_sq_3_f":1,
    "T_sq_4_nf":1,
}

sensor_sheets = ["Accelerometer", 'Gyroscope', 'Linear Acceleration', 'Orientation']

for file_name, cut in CUTS.items():
    input_file = f"preproc/{file_name}.xlsx"
    output_file = f"preproc/{file_name}.xlsx"

    with pd.ExcelFile(input_file) as xls:
        sheet_data = {}

        for sheet in xls.sheet_names:
            if sheet not in sensor_sheets:
                continue
            df = pd.read_excel(xls, sheet_name=sheet)

            if sheet in sensor_sheets:
                df = df[
                    (df["Time (s)"] >= cut)
                    & (df["Time (s)"] < cut + 30)
                ].copy()

            sheet_data[sheet] = df

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for sheet, df in sheet_data.items():
            df.to_excel(
                writer,
                sheet_name=sheet,
                index=False
            )

    print(f"Processed {file_name}")

Processed S_push_1_f
Processed S_push_2_nf
Processed S_push_3_f
Processed S_push_4_nf
Processed S_row_1_f
Processed S_row_2_nf
Processed S_row_3_f
Processed S_row_4_nf
Processed S_sq_1_f
Processed S_sq_2_nf
Processed S_sq_3_f
Processed S_sq_4_nf
Processed T_push_1_f
Processed T_push_2_nf
Processed T_push_3_f
Processed T_push_4_nf
Processed T_row_1_f
Processed T_row_2_nf
Processed T_row_3_f
Processed T_row_4_nf
Processed T_sq_1_f
Processed T_sq_2_nf
Processed T_sq_3_f
Processed T_sq_4_nf


In [60]:
encoder = {
    "subject": {"S": 0, "T": 1},
    "exercise": {"push": 0, "row": 1, "sq": 2},
    "focus": {"nf": 0, "f": 1},
}
files = ["S_push_1_f", "S_push_2_nf", "S_push_3_f", "S_push_4_nf",
          "S_row_1_f", "S_row_2_nf", "S_row_3_f", "S_row_4_nf",
          "S_sq_1_f", "S_sq_2_nf", "S_sq_3_f", "S_sq_4_nf",
          "T_push_1_f", "T_push_2_nf", "T_push_3_f", "T_push_4_nf",
          "T_row_1_f", "T_row_2_nf", "T_row_3_f", "T_row_4_nf",
          "T_sq_1_f", "T_sq_2_nf", "T_sq_3_f", "T_sq_4_nf",]

sensor_sheets = ["Accelerometer", 'Gyroscope', 'Linear Acceleration', 'Orientation']

files_union = None
for file in files:
    sensors_joined = None
    for sensor in sensor_sheets:
        df_sensor = pd.read_excel(f"data/{file}.xlsx", sheet_name=sensor)

        df_sensor_resampled = resample_by_sensor(df_sensor, sensor)

        if sensors_joined is None:
            sensors_joined = df_sensor_resampled.copy()
        else:
            sensors_joined = pd.merge(sensors_joined, df_sensor_resampled, on="Time (s)")

    exercise_data = file.split("_")
    subject = exercise_data[0]
    exercise = exercise_data[1]
    set_nr = exercise_data[2]
    focus = exercise_data[3]

    sensors_joined["subject"] = encoder["subject"][subject]
    sensors_joined["exercise"] = encoder["exercise"][exercise]
    sensors_joined["set_nr"] = int(set_nr)
    sensors_joined["focus"] = encoder["focus"][focus]

    if files_union is None:
        files_union = sensors_joined.copy()
    else:
        files_union = pd.concat([files_union, sensors_joined], ignore_index=True)

files_union = files_union.rename(
    columns={
        "Time (s)": "time",
        "Gyroscope x (rad/s)": "gyro_x",
        "Gyroscope y (rad/s)": "gyro_y",
        "Gyroscope z (rad/s)": "gyro_z",
        "Linear Acceleration x (m/s^2)": "acc_lin_x",
        "Linear Acceleration y (m/s^2)": "acc_lin_y",
        "Linear Acceleration z (m/s^2)": "acc_lin_z",
        "Acceleration x (m/s^2)": "acc_x",
        "Acceleration y (m/s^2)": "acc_y",
        "Acceleration z (m/s^2)": "acc_z",
        "Yaw (°)": "yaw",
        "Pitch (°)": "pitch",
        "Roll (°)": "roll",
    }
)
files_union.to_csv("data/phyphox_df")